# Examen bimestral

**Introducción a los Sistemas de Información**

Prof. Iván Carrera

Ingeniería en Ciencia de Datos e Inteligencia Artificial · Escuela Politécnica Nacional

**Calificación total:** 10 puntos.

## Caso de estudio · Papelería universitaria

Una papelería universitaria registra las compras de sus sucursales Central, Norte y Sur en un sistema de punto de venta. Cada compra genera un ticket que puede incluir varios productos. La persona encargada de la papelería necesita resumir estos registros para conocer qué se vendió, cómo se pagaron las compras y cómo variaron las ventas entre sucursales y meses.

Elabore los reportes solicitados a partir de los datos.

## Archivos de trabajo

| Archivo      | Contenido |
|--------------|---|
| `productos.csv` | Una fila por producto. Incluye su código, nombre, categoría y precio de lista. |
| `clientes.csv` | Una fila por cliente. Incluye su identificador, nombre, ciudad y tipo de cliente. |
| `ventas.csv`   | Una fila por producto dentro de un ticket. Incluye la fecha, sucursal, cliente, cantidad, precio unitario y método de pago. |

El campo `id_ticket` puede repetirse en varias filas de `ventas.csv`. Para relacionar los archivos, use `id_producto` en ventas y productos, y `ventas.id_cliente` con `clientes.id`. Un campo vacío en `metodo_pago` indica que ese dato quedó sin registrar.

## Preguntas

1. **Líneas, tickets y unidades.** ¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.

2. **Tickets por sucursal.** ¿Cuántos tickets se registraron en cada sucursal? ¿Cuál registró más?

3. **Métodos de pago.** ¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?

4. **Precios de lista por categoría.** ¿Cuál es el precio de lista mínimo, máximo y promedio de los productos de cada categoría? Use el catálogo de productos.

5. **Unidades por sucursal y categoría.** ¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de `cantidad` en `ventas.csv`.

6. **Importe de ventas por sucursal.** ¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como `cantidad × precio_unitario`.

7. **Tamaño de las compras.** ¿Cuántos tickets corresponden a compras de **hasta 7 unidades**, **de 8 a 14 unidades** y **de 15 o más unidades**? Sume primero las cantidades de todas las líneas de cada ticket.

8. **Ventas por mes.** ¿Cuántos tickets y qué importe total se registraron en cada mes? ¿En qué mes se alcanzó el mayor importe?

9. **Producto más vendido por sucursal.** ¿Qué producto acumuló más unidades vendidas en cada sucursal? Indique su código, nombre y cantidad vendida. Si hay un empate, incluya todos los productos empatados.

10. **Compras por tipo de cliente.** Para cada tipo de cliente, ¿cuántos clientes distintos compraron, cuántos tickets generaron y cuál fue el importe total de sus compras?

## Entrega y evaluación

Resuelva los reportes solicitados en un mismo Jupyter Notebook, identificados con el número de cada pregunta. Presente los reportes con encabezados claros y los importes en dólares con dos decimales. Use conteos, agrupaciones y cálculos descriptivos.

Entregue el link al Jupyter Notebook cargado en un repositorio Git en la Tarea del aula virtual.

**Cada pregunta vale 1 punto, para un total de 10 puntos.** En cada respuesta se evaluará que el código produzca el resultado solicitado a partir de los CSV y que se presenten todos los elementos pedidos en la pregunta.

Buena suerte.

## Preguntas

In [ ]:
import pandas as pd

clientes = pd.read_csv("clientes.csv")
productos = pd.read_csv("productos.csv")
ventas = pd.read_csv("ventas.csv")

1. **Líneas, tickets y unidades.** ¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.

In [ ]:
lineas_venta = len(ventas)
tickets_distintos = ventas["id_ticket"].nunique()
unidades_vendidas = ventas["cantidad"].sum()

print("Resumen general de ventas")
print("-----------------------")
print("La cantidad de líneas de venta es:", lineas_venta)
print("La cantidad de tickets distintos es:", tickets_distintos)
print("La cantidad de unidades vendidas es:", unidades_vendidas)

**Explicación de cada conteo:**
- **Líneas de venta:** Representa el total de filas registradas en `ventas.csv`; cada línea corresponde a un producto distinto dentro de un ticket.
- **Tickets distintos:** Representa el total de compras únicas realizadas (`id_ticket` sin repetir), ya que un mismo ticket puede incluir varios productos.
- **Unidades vendidas:** Representa la suma total de artículos vendidos en todas las líneas de venta (suma de la columna `cantidad`).

2. **Tickets por sucursal.** ¿Cuántos tickets se registraron en cada sucursal? ¿Cuál registró más?

In [ ]:
tickets_por_sucursal = ventas.groupby("sucursal")["id_ticket"].nunique()

print("Los tickets registrados por sucursal son:")
print(tickets_por_sucursal)
print("-----------------------")
print("La sucursal con más tickets es:", tickets_por_sucursal.idxmax(), "con", tickets_por_sucursal.max(), "tickets")

3. **Métodos de pago.** ¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?

In [ ]:
metodos = ventas["metodo_pago"].fillna("Sin registrar")
tickets_por_metodo = ventas.groupby(metodos)["id_ticket"].nunique()

print("Los tickets pagados por cada método son:")
print(tickets_por_metodo)
print("-----------------------")
print("Tickets sin método de pago registrado:", tickets_por_metodo["Sin registrar"])

4. **Precios de lista por categoría.** ¿Cuál es el precio de lista mínimo, máximo y promedio de los productos de cada categoría? Use el catálogo de productos.

In [ ]:
precios_categoria = productos.groupby("categoria")["precio_lista"].agg(
    Minimo_USD="min",
    Maximo_USD="max",
    Promedio_USD="mean"
).round(2)

print("Precios de lista por categoría (en dólares):")
print(precios_categoria)

5. **Unidades por sucursal y categoría.** ¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de `cantidad` en `ventas.csv`.

In [ ]:
ventas_con_categoria = pd.merge(ventas, productos, on="id_producto", how="inner")

tabla_unidades = pd.crosstab(
    ventas_con_categoria["sucursal"],
    ventas_con_categoria["categoria"],
    values=ventas_con_categoria["cantidad"],
    aggfunc="sum",
    margins=True,
    margins_name="Total"
)

print("Unidades vendidas por sucursal y categoría:")
print(tabla_unidades)
print("-----------------------")

total_tabla = tabla_unidades.loc["Total", "Total"]
total_ventas = ventas["cantidad"].sum()
print("Total general en la tabla:", total_tabla)
print("Suma de cantidad en ventas.csv:", total_ventas)
print("¿Coinciden los totales?:", total_tabla == total_ventas)

6. **Importe de ventas por sucursal.** ¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como `cantidad × precio_unitario`.

In [ ]:
ventas["subtotal"] = ventas["cantidad"] * ventas["precio_unitario"]
importe_por_sucursal = ventas.groupby("sucursal")["subtotal"].sum().round(2)

print("Importe total de ventas por sucursal (en dólares):")
for sucursal, total in importe_por_sucursal.items():
    print(f"El importe de la sucursal {sucursal} es: ${total:.2f}")

7. **Tamaño de las compras.** ¿Cuántos tickets corresponden a compras de **hasta 7 unidades**, **de 8 a 14 unidades** y **de 15 o más unidades**? Sume primero las cantidades de todas las líneas de cada ticket.

In [ ]:
unidades_por_ticket = ventas.groupby("id_ticket")["cantidad"].sum()
grupo_tickets = pd.cut(
    unidades_por_ticket,
    bins=[-1, 7, 14, float("inf")],
    labels=["Hasta 7 unidades", "De 8 a 14 unidades", "De 15 o más unidades"]
)

print("Cantidad de tickets según el tamaño de compra:")
print(grupo_tickets.value_counts(sort=False))

8. **Ventas por mes.** ¿Cuántos tickets y qué importe total se registraron en cada mes? ¿En qué mes se alcanzó el mayor importe?

In [ ]:
ventas["subtotal"] = ventas["cantidad"] * ventas["precio_unitario"]
ventas["mes"] = pd.to_datetime(ventas["fecha"]).dt.to_period("M")

ventas_por_mes = ventas.groupby("mes").agg(
    Tickets=("id_ticket", "nunique"),
    Importe_USD=("subtotal", "sum")
).round(2)

print("Tickets e importe total por mes (en dólares):")
print(ventas_por_mes)
print("-----------------------")

mes_mayor = ventas_por_mes["Importe_USD"].idxmax()
importe_mayor = ventas_por_mes["Importe_USD"].max()
print(f"El mes con mayor importe fue {mes_mayor} con ${importe_mayor:.2f}")

9. **Producto más vendido por sucursal.** ¿Qué producto acumuló más unidades vendidas en cada sucursal? Indique su código, nombre y cantidad vendida. Si hay un empate, incluya todos los productos empatados.

In [ ]:
ventas_prod = pd.merge(ventas, productos, on="id_producto")
totales = ventas_prod.groupby(["sucursal", "id_producto", "producto"], as_index=False)["cantidad"].sum()

print("Producto más vendido por sucursal:")
for sucursal in totales["sucursal"].unique():
    datos_sucursal = totales[totales["sucursal"] == sucursal]
    max_cantidad = datos_sucursal["cantidad"].max()
    mas_vendidos = datos_sucursal[datos_sucursal["cantidad"] == max_cantidad]
    print("-----------------------")
    print("Sucursal:", sucursal)
    print(mas_vendidos[["id_producto", "producto", "cantidad"]].to_string(index=False))

10. **Compras por tipo de cliente.** Para cada tipo de cliente, ¿cuántos clientes distintos compraron, cuántos tickets generaron y cuál fue el importe total de sus compras?

In [ ]:
ventas["subtotal"] = ventas["cantidad"] * ventas["precio_unitario"]
ventas_clientes = pd.merge(ventas, clientes, left_on="id_cliente", right_on="id", how="inner")

resumen_clientes = ventas_clientes.groupby("tipo_cliente").agg(
    Clientes_Distintos=("id_cliente", "nunique"),
    Tickets=("id_ticket", "nunique"),
    Importe_Total_USD=("subtotal", "sum")
).round(2)

print("Compras por tipo de cliente (importes en dólares):")
print(resumen_clientes)